# Southern OC + LA/OC/SD Event Finder (Next 4 Weeks)

This notebook pulls **concerts**, **plays/theatre**, and other **events** that are likely to interest **teens and adults** — and tries to avoid:

- Events meant primarily for **babies/young kids**
- **Lead‑gen / “ad disguised as an event”** (free trials, open houses, info sessions, etc.)
- Low‑signal **nightclub / DJ nights** and **restaurant promos**

It also includes a curated **major venue concert pass** (LA / OC / San Diego) and highlights a **Major Artists** section (based on Wikipedia popularity signals).

**Default time window:** the **next 4 weeks starting tomorrow** (relative to the run date in the configured timezone).

> Scraping note: this notebook uses a mix of official APIs and light HTML parsing. Respect site terms/robots and keep request volume low.


In [ ]:
# Dependencies
# - In Google Colab, this notebook will install required packages automatically.
# - In GitHub Actions / local runs, install via requirements.txt (so we skip installs here).

import sys
import subprocess
import os
import re
import math
import json
import time
from dataclasses import dataclass, asdict
from datetime import datetime, date, timedelta, time as dtime
from zoneinfo import ZoneInfo
from typing import Optional, List, Dict, Any, Tuple

def _pip_install(pkgs: List[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    _pip_install([
        "requests",
        "beautifulsoup4",
        "pandas",
        "python-dateutil",
        "geopy",
        "tqdm",
        "tenacity",
        "diskcache",
        "lxml",
    ])

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dateutil import parser as dateparser
from geopy.distance import geodesic
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from diskcache import Cache

pd.set_option("display.max_colwidth", 120)

# Small on-disk cache (persists during runtime; helps avoid re-hitting Wikipedia/geocoding repeatedly)
cache = Cache("./.cache_event_finder")


## 1) Configuration

Set your **home base** (used for distance ranking) and optionally add API keys.

- **Ticketmaster Discovery API key (recommended):** sign up for a free key and set it below.
- Optional: **Google Places API key** can be used to fetch venue rating counts (kept optional because it requires billing).


In [ ]:
#@title Configuration

import os

# Timezone used for window + display
TZ = "America/Los_Angeles"  #@param {type:"string"}

# Southern Orange County "home base" (used for distance scoring; events without coords are still shown)
HOME_LABEL = "Laguna Niguel, CA"
HOME_LAT = 33.5225   #@param {type:"number"}
HOME_LON = -117.7076 #@param {type:"number"}

# How far out to search for the radius-based Ticketmaster pull (miles)
RADIUS_MILES = 45  #@param {type:"integer"}

# Default window: next N weeks starting tomorrow (relative to run date in TZ)
WINDOW_WEEKS = 4  #@param {type:"integer"}

# Optional manual override (useful for debugging)
USE_MANUAL_DATES = False  #@param {type:"boolean"}
MANUAL_START = "2025-12-24"  #@param {type:"string"}  # YYYY-MM-DD
MANUAL_END   = "2026-01-20"  #@param {type:"string"}  # YYYY-MM-DD

# Ticketmaster Discovery API (recommended primary source)
TICKETMASTER_API_KEY = ""  #@param {type:"string"}

# Optional: Google Places (venue ratings & review counts)
GOOGLE_PLACES_API_KEY = ""  #@param {type:"string"}

# Also pull concerts from a curated list of major venues in LA / OC / San Diego.
INCLUDE_MAJOR_VENUE_CONCERTS = True  #@param {type:"boolean"}

# Major-artist detection (Wikipedia popularity score in 0..1; higher = more popular)
MAJOR_ARTIST_FAME_THRESHOLD = 0.65  #@param {type:"number"}

# Hard excludes (venue blacklist)
EXCLUDED_VENUE_SUBSTRINGS = [
    "Hamburger Mary",  # remove anything from Hamburger Mary's
]

# Optional: guard so a daily schedule only produces output on Tuesdays
ENABLE_TUESDAY_GUARD = False  #@param {type:"boolean"}

# ---------- Env overrides (useful for GitHub Actions / secrets) ----------
TZ = os.getenv("TZ", TZ)

HOME_LABEL = os.getenv("HOME_LABEL", HOME_LABEL)
HOME_LAT = float(os.getenv("HOME_LAT", HOME_LAT))
HOME_LON = float(os.getenv("HOME_LON", HOME_LON))
RADIUS_MILES = int(os.getenv("RADIUS_MILES", RADIUS_MILES))

WINDOW_WEEKS = int(os.getenv("WINDOW_WEEKS", WINDOW_WEEKS))

USE_MANUAL_DATES = os.getenv("USE_MANUAL_DATES", str(USE_MANUAL_DATES)).lower() in ("1", "true", "yes", "y")
MANUAL_START = os.getenv("MANUAL_START", MANUAL_START)
MANUAL_END = os.getenv("MANUAL_END", MANUAL_END)

TICKETMASTER_API_KEY = os.getenv("TICKETMASTER_API_KEY", TICKETMASTER_API_KEY).strip()
GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_PLACES_API_KEY", GOOGLE_PLACES_API_KEY).strip()

INCLUDE_MAJOR_VENUE_CONCERTS = os.getenv("INCLUDE_MAJOR_VENUE_CONCERTS", str(INCLUDE_MAJOR_VENUE_CONCERTS)).lower() in ("1", "true", "yes", "y")
MAJOR_ARTIST_FAME_THRESHOLD = float(os.getenv("MAJOR_ARTIST_FAME_THRESHOLD", MAJOR_ARTIST_FAME_THRESHOLD))

ENABLE_TUESDAY_GUARD = os.getenv("ENABLE_TUESDAY_GUARD", str(ENABLE_TUESDAY_GUARD)).lower() in ("1", "true", "yes", "y")


In [ ]:
# Optional: guard so a daily schedule only produces output on Tuesday
if ENABLE_TUESDAY_GUARD:
    run_date = datetime.now(ZoneInfo(TZ)).date()
    if run_date.weekday() != 1:  # Tuesday
        print(f"Not Tuesday ({run_date:%A}); exiting early.")
        raise SystemExit
    print("Tuesday run: continuing.")
else:
    print("Tuesday guard disabled (normal interactive runs).")


In [ ]:
def compute_next_weeks_window(run_date: Optional[date] = None, weeks: int = 4, tz: str = TZ) -> Tuple[datetime, datetime]:
    """Return (start_dt, end_dt) for the next `weeks` weeks starting tomorrow, in local tz.

    Example: run on Tue -> starts Wed (next day) and spans 4 weeks.
    """
    if run_date is None:
        run_date = datetime.now(ZoneInfo(tz)).date()

    start_date = run_date + timedelta(days=1)
    # Inclusive end: last second of the final day
    end_date = start_date + timedelta(days=(7 * int(weeks)) - 1)

    start_dt = datetime.combine(start_date, dtime(0, 0), ZoneInfo(tz))
    end_dt = datetime.combine(end_date, dtime(23, 59, 59), ZoneInfo(tz))
    return start_dt, end_dt

def get_target_window() -> Tuple[datetime, datetime]:
    if USE_MANUAL_DATES:
        s = dateparser.parse(MANUAL_START).date()
        e = dateparser.parse(MANUAL_END).date()
        return (
            datetime.combine(s, dtime(0,0), ZoneInfo(TZ)),
            datetime.combine(e, dtime(23,59,59), ZoneInfo(TZ)),
        )
    return compute_next_weeks_window(weeks=WINDOW_WEEKS)

START_DT, END_DT = get_target_window()
print(f"Target window (local): {START_DT}  →  {END_DT}")


In [ ]:
MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}

def _clean_ordinals(s: str) -> str:
    return re.sub(r"(\d)(st|nd|rd|th)\b", r"\1", s, flags=re.I)

def parse_date_span(text: str, default_year: Optional[int] = None) -> Tuple[Optional[date], Optional[date]]:
    """Parse strings like:
      - 'December 6 - 28, 2025'
      - 'January 14 – February 1, 2026'
      - 'Feb. 20-Mar. 8, 2026'
      - 'December 31, 2025'
    Returns (start_date, end_date).
    """
    if not text:
        return None, None
    t = _clean_ordinals(text)
    t = t.replace("\u2013", "-").replace("\u2014", "-")  # en/em dash
    t = t.replace("–", "-").replace("—", "-")
    t = re.sub(r"\s+", " ", t).strip()
    # common connectors
    t = t.replace(" to ", " - ")

    # If no dash, parse single date
    if "-" not in t:
        try:
            d = dateparser.parse(t, fuzzy=True, default=datetime(default_year or datetime.now().year, 1, 1)).date()
            return d, d
        except Exception:
            return None, None

    left, right = [x.strip() for x in t.split("-", 1)]

    # If right has no month, inherit from left
    left_m = re.search(r"\b([A-Za-z]{3,9})\b", left)
    right_m = re.search(r"\b([A-Za-z]{3,9})\b", right)
    if left_m and not right_m:
        # prepend month name
        right = f"{left_m.group(1)} {right}"

    # If right has no year, inherit from left if present
    left_y = re.search(r"\b(20\d{2}|19\d{2})\b", left)
    right_y = re.search(r"\b(20\d{2}|19\d{2})\b", right)
    if left_y and not right_y:
        right = f"{right}, {left_y.group(1)}"

    # Try parse both
    try:
        sd = dateparser.parse(left, fuzzy=True, default=datetime(default_year or datetime.now().year, 1, 1)).date()
    except Exception:
        sd = None
    try:
        ed = dateparser.parse(right, fuzzy=True, default=datetime(default_year or datetime.now().year, 1, 1)).date()
    except Exception:
        ed = None

    return sd, ed

def parse_time_text(t: str) -> Optional[dtime]:
    if not t:
        return None
    try:
        # dateutil can parse time-only if we give a default date
        dt = dateparser.parse(t, fuzzy=True, default=datetime(2000,1,1,0,0))
        return dt.time()
    except Exception:
        return None

def overlaps_window(start: Optional[datetime], end: Optional[datetime], window_start: datetime, window_end: datetime) -> bool:
    if start is None:
        return False
    if end is None:
        end = start
    return (start <= window_end) and (end >= window_start)

@dataclass
class Event:
    title: str
    category: str  # 'Concert', 'Play', 'Event'
    start: Optional[datetime]
    end: Optional[datetime]
    venue: Optional[str] = None
    city: Optional[str] = None
    region: Optional[str] = None
    address: Optional[str] = None
    lat: Optional[float] = None
    lon: Optional[float] = None
    source: Optional[str] = None
    url: Optional[str] = None
    description: Optional[str] = None
    price_min: Optional[float] = None
    price_max: Optional[float] = None
    age_hint: Optional[str] = None  # e.g., '12+'
    tags: Optional[str] = None

def event_to_row(e: Event) -> Dict[str, Any]:
    d = asdict(e)
    return d


In [ ]:
# Known venue coordinates (helps avoid geocoding and improves proximity scoring)

KNOWN_VENUES = {
    # Southern OC / nearby
    "South Coast Repertory": (33.6897, -117.8852),  # Costa Mesa (approx)
    "Laguna Playhouse": (33.5422, -117.7845),       # Laguna Beach (approx)
    "Renée and Henry Segerstrom Concert Hall": (33.6902, -117.8880),  # Segerstrom Center campus
    "Samueli Theater": (33.6902, -117.8880),
    "Segerstrom Stage": (33.6897, -117.8852),
    "Julianne Argyros Stage": (33.6897, -117.8852),
    "Soka Performing Arts Center": (33.5696, -117.7251),  # Aliso Viejo (approx)
    "House of Blues Anaheim": (33.8036, -117.9133),
    "The Parish at House of Blues Anaheim": (33.8036, -117.9133),
    "The Observatory": (33.7133, -117.9236),  # Santa Ana
    "Constellation Room": (33.7133, -117.9236),
    "Irvine Barclay Theatre": (33.6455, -117.8433),
    "Honda Center": (33.8070, -117.8766),

    # LA / San Diego "major venues" (approx; used as fallback if event coords are missing)
    "The Observatory North Park": (32.7493, -117.1297),
    "Kia Forum": (33.9583, -118.3417),
    "The Forum": (33.9583, -118.3417),
    "Los Angeles Memorial Coliseum": (34.0141, -118.2879),
    "Hollywood Palladium": (34.0982, -118.3257),
    "The Fonda Theatre": (34.1017, -118.3288),
    "Henry Fonda Theater": (34.1017, -118.3288),
    "Troubadour": (34.0839, -118.3882),
    "Hollywood Bowl": (34.1122, -118.3391),
    "Greek Theatre": (34.1193, -118.3004),  # LA
    "The Wiltern": (34.0616, -118.3080),
    "Dolby Theatre": (34.1023, -118.3407),
    "Walt Disney Concert Hall": (34.0553, -118.2498),
    "The Bellwether": (34.0410, -118.2519),
}

# Venue prominence: 0..1 (used for "compelling" score and for filtering low-signal nightlife promos)
VENUE_PROMINENCE = {
    # OC
    "Renée and Henry Segerstrom Concert Hall": 0.92,
    "Samueli Theater": 0.88,
    "Segerstrom Center": 0.90,
    "South Coast Repertory": 0.90,
    "Irvine Barclay Theatre": 0.86,
    "Laguna Playhouse": 0.84,
    "Honda Center": 0.92,
    "House of Blues Anaheim": 0.84,
    "The Observatory": 0.86,
    "Soka Performing Arts Center": 0.82,

    # LA / SD major venues
    "The Observatory North Park": 0.86,
    "Kia Forum": 0.96,
    "Los Angeles Memorial Coliseum": 0.96,
    "Hollywood Palladium": 0.90,
    "The Fonda Theatre": 0.88,
    "Troubadour": 0.88,
    "Hollywood Bowl": 0.98,
    "Greek Theatre": 0.93,
    "The Wiltern": 0.90,
    "Dolby Theatre": 0.93,
    "Walt Disney Concert Hall": 0.95,
    "The Bellwether": 0.88,
}

# Curated list for the "major venues" concert pass (Ticketmaster venueId lookup + venueId event search)
MAJOR_CONCERT_VENUES = [
    {"label": "The Observatory (Santa Ana)", "keyword": "The Observatory", "city": "Santa Ana", "stateCode": "CA",
     "aliases": ["Observatory OC", "The Observatory OC", "Constellation Room"]},
    {"label": "The Observatory North Park", "keyword": "The Observatory North Park", "city": "San Diego", "stateCode": "CA",
     "aliases": ["Observatory North Park"]},
    {"label": "Honda Center", "keyword": "Honda Center", "city": "Anaheim", "stateCode": "CA", "aliases": []},
    {"label": "Kia Forum", "keyword": "Kia Forum", "city": "Inglewood", "stateCode": "CA", "aliases": ["The Forum", "The Kia Forum"]},
    {"label": "Los Angeles Memorial Coliseum", "keyword": "Los Angeles Memorial Coliseum", "city": "Los Angeles", "stateCode": "CA",
     "aliases": ["LA Memorial Coliseum", "The Coliseum"]},
    {"label": "Hollywood Palladium", "keyword": "Hollywood Palladium", "city": "Hollywood", "stateCode": "CA", "aliases": []},
    {"label": "The Fonda Theatre", "keyword": "Fonda Theatre", "city": "Los Angeles", "stateCode": "CA",
     "aliases": ["The Fonda Theatre", "Henry Fonda Theater", "Henry Fonda Theatre"]},
    {"label": "Troubadour", "keyword": "Troubadour", "city": "West Hollywood", "stateCode": "CA", "aliases": []},
    {"label": "Hollywood Bowl", "keyword": "Hollywood Bowl", "city": "Hollywood", "stateCode": "CA", "aliases": []},
    {"label": "Greek Theatre (Los Angeles)", "keyword": "Greek Theatre", "city": "Los Angeles", "stateCode": "CA",
     "aliases": ["The Greek Theatre"]},
    {"label": "The Wiltern", "keyword": "Wiltern", "city": "Los Angeles", "stateCode": "CA", "aliases": ["The Wiltern"]},
    {"label": "Dolby Theatre", "keyword": "Dolby Theatre", "city": "Hollywood", "stateCode": "CA", "aliases": []},
    {"label": "Walt Disney Concert Hall", "keyword": "Walt Disney Concert Hall", "city": "Los Angeles", "stateCode": "CA",
     "aliases": ["Disney Concert Hall"]},
    {"label": "The Bellwether", "keyword": "Bellwether", "city": "Los Angeles", "stateCode": "CA", "aliases": ["The Bellwether"]},
]

def infer_coords(venue: Optional[str], address: Optional[str] = None) -> Tuple[Optional[float], Optional[float]]:
    if venue:
        for k, (lat, lon) in KNOWN_VENUES.items():
            if k.lower() in venue.lower():
                return lat, lon
        if venue in KNOWN_VENUES:
            return KNOWN_VENUES[venue]
    return None, None


In [ ]:
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type(Exception),
)
def geocode_address(addr: str) -> Tuple[Optional[float], Optional[float]]:
    """Geocode an address using Nominatim (OpenStreetMap)."""
    if not addr:
        return None, None
    cache_key = f"geocode::{addr}"
    if cache_key in cache:
        return cache[cache_key]

    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": addr, "format": "json", "limit": 1}
    headers = {"User-Agent": "soc-weekly-event-finder/1.0 (personal use)"}
    r = requests.get(url, params=params, headers=headers, timeout=20)
    r.raise_for_status()
    data = r.json()
    if not data:
        cache[cache_key] = (None, None)
        return None, None
    lat, lon = float(data[0]["lat"]), float(data[0]["lon"])
    cache[cache_key] = (lat, lon)
    # be polite
    time.sleep(1.0)
    return lat, lon


In [ ]:
# Heuristic filters
KIDS_KEYWORDS = [
    "baby", "babies", "toddler", "toddlers", "preschool", "pre-school", "storybook", "storytime",
    "sensory", "mommy", "daddy", "parent & me", "parent-and-me", "kids", "kid-friendly",
    "children", "childrens", "little ones", "ages 0", "ages 1", "ages 2", "ages 3", "ages 4",
    "kindergarten", "playdate",
]

PROMO_KEYWORDS = [
    "open house", "free trial", "trial class", "intro class", "information session", "info session",
    "seminar", "webinar", "workshop", "bootcamp", "enroll", "sign up", "signup", "membership",
    "networking", "mixer", "sales pitch", "consultation", "demo", "sample", "grand opening",
    "vendor fair", "market your", "leadership training", "real estate", "insurance",
]

NIGHTLIFE_KEYWORDS = [
    "dj", "dance party", "rave", "after party", "afterparty", "club night", "ladies night",
    "bottle service", "guest list", "happy hour", "karaoke", "open mic", "brunch party",
    "bar crawl", "pub crawl", "kpop night", "taylor swift night", "beyonce night",
]

LOW_SIGNAL_KEYWORDS = [
    "parking", "membership", "mailing list", "meet & greet", "meet and greet", "vip table",
]

def _txt(e: Event) -> str:
    parts = [e.title or "", e.venue or "", e.description or "", e.tags or "", e.age_hint or ""]
    return " ".join([p for p in parts if p]).lower()

def is_kids_event(e: Event) -> bool:
    t = _txt(e)
    if any(k in t for k in KIDS_KEYWORDS):
        # allow if it's clearly teen-appropriate (rare). Keep it simple: exclude.
        return True
    # If we have explicit age hint like '4+' treat as kids.
    if e.age_hint:
        m = re.search(r"(\d{1,2})\s*\+", e.age_hint)
        if m and int(m.group(1)) <= 8:
            return True
    return False

def is_promo_event(e: Event) -> bool:
    t = _txt(e)
    return any(k in t for k in PROMO_KEYWORDS)

def is_nightlife_event(e: Event) -> bool:
    t = _txt(e)
    return any(k in t for k in NIGHTLIFE_KEYWORDS)

def is_low_signal(e: Event) -> bool:
    t = _txt(e)
    return any(k in t for k in LOW_SIGNAL_KEYWORDS)


In [ ]:
def distance_miles(lat: float, lon: float) -> float:
    return geodesic((HOME_LAT, HOME_LON), (lat, lon)).miles

def proximity_score(dist_mi: Optional[float]) -> float:
    """Proximity score in [0, 100]. Unknown distance returns 0."""
    if dist_mi is None:
        return 0.0
    try:
        if isinstance(dist_mi, float) and math.isnan(dist_mi):
            return 0.0
        dist = float(dist_mi)
    except Exception:
        return 0.0
    # 0 miles -> 100, 50 miles -> ~0
    return max(0.0, 100.0 - (dist * 2.0))

def venue_prominence(venue: Optional[str]) -> float:
    if not venue:
        return 0.4
    best = 0.4
    for k, v in VENUE_PROMINENCE.items():
        if k.lower() in venue.lower():
            best = max(best, v)
    return best

def _clean_title_for_wiki(title: str) -> str:
    t = (title or "").strip()
    t = re.sub(r"\s*\(.*?\)\s*$", "", t).strip()
    t = re.sub(r"\s+", " ", t).strip()
    return t

def guess_primary_entity_for_wiki(e: Event) -> str:
    """Best-effort guess at the 'primary' artist/entity to use for Wikipedia popularity.

    For concerts, Ticketmaster titles are often:
      - 'Artist Name'
      - 'Artist Name: Tour Name'
      - 'Artist Name w/ Opener'
    We try to extract the likely headliner.
    """
    t = _clean_title_for_wiki(e.title or "")
    if not t:
        return ""

    if (e.category or "").lower() == "concert":
        # Prefer text before ':' if it looks like an artist name
        if ":" in t:
            left = t.split(":", 1)[0].strip()
            if 2 <= len(left) <= 70:
                t = left

        # Split common "with/feat" patterns
        splitters = [" w/ ", " with ", " featuring ", " feat. ", " feat ", " presents "]
        tl = " " + t.lower() + " "
        for s in splitters:
            if s in tl:
                # split on the *first* occurrence in original string (case-insensitive)
                parts = re.split(re.escape(s.strip()), t, flags=re.I)
                if parts and parts[0].strip():
                    t = parts[0].strip()
                break

        # If there's a " + " or " & " list, take the first item (headliner-ish)
        for s in [" + ", " & ", " / "]:
            if s in t:
                left = t.split(s, 1)[0].strip()
                if 2 <= len(left) <= 70:
                    t = left

    return t.strip()

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type(requests.RequestException),
)
def wikipedia_pageviews_score(query: str, days: int = 30) -> float:
    """Return a 0..1 fame score using Wikipedia pageviews (optional)."""
    q = (query or "").strip()
    if not q:
        return 0.0

    cache_key = f"wiki::{q.lower()}::{days}"
    if cache_key in cache:
        return cache[cache_key]

    # 1) Search
    s_url = "https://en.wikipedia.org/w/api.php"
    s_params = {
        "action": "query",
        "list": "search",
        "srsearch": q,
        "format": "json",
        "srlimit": 1,
    }
    s = requests.get(s_url, params=s_params, timeout=15, headers={"User-Agent": "soc-event-finder/1.0"}).json()
    hits = (s.get("query", {}) or {}).get("search", []) or []
    if not hits:
        cache[cache_key] = 0.0
        return 0.0

    title = hits[0].get("title")
    if not title:
        cache[cache_key] = 0.0
        return 0.0

    # 2) Pageviews (last N days)
    # Wikimedia REST API expects spaces -> _
    t_enc = title.replace(" ", "_")
    end = datetime.utcnow().date() - timedelta(days=1)
    start = end - timedelta(days=days)
    pv_url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/{t_enc}/daily/{start:%Y%m%d}/{end:%Y%m%d}"
    r = requests.get(pv_url, timeout=15, headers={"User-Agent": "soc-event-finder/1.0"})
    if r.status_code != 200:
        cache[cache_key] = 0.0
        return 0.0
    js = r.json()
    items = js.get("items", []) or []
    total = sum(int(it.get("views") or 0) for it in items)

    # Log-ish normalization: map to 0..1 with a soft cap.
    # ~50k views/month -> ~0.6; ~200k views/month -> ~0.8; ~1M+ -> ~1.0
    score = 0.0
    try:
        score = max(0.0, min(1.0, math.log10(1 + total) / 6.0))
    except Exception:
        score = 0.0

    cache[cache_key] = score
    return score

def nightlife_penalty(e: Event) -> float:
    t = _txt(e)
    p = 0.0
    if "tribute" in t:
        p += 0.10
    if is_nightlife_event(e):
        p += 0.45
    if is_low_signal(e):
        p += 0.35
    return min(0.9, p)

def compelling_score(e: Event, fame_override: Optional[float] = None, use_wikipedia: bool = True) -> Tuple[float, float]:
    """Return (compelling_0_to_100, fame_0_to_1).

    Concerts weight 'fame' more heavily to prioritize well-known acts.
    """
    base = venue_prominence(e.venue)  # 0..1
    src = 0.85 if (e.source or "").lower() in ["ticketmaster", "livenation", "pacificsymphony", "scr", "lagunaplayhouse"] else 0.65

    fame = 0.0
    if fame_override is not None:
        fame = float(fame_override)
    elif use_wikipedia:
        q = guess_primary_entity_for_wiki(e)
        fame = wikipedia_pageviews_score(q)

    # Prioritize well-known acts (especially for concerts)
    if (e.category or "").lower() == "concert":
        comp01 = (0.25 * base) + (0.20 * src) + (0.55 * fame)
    else:
        comp01 = (0.45 * base) + (0.20 * src) + (0.35 * fame)

    comp01 = max(0.0, min(1.0, comp01 - nightlife_penalty(e)))
    return comp01 * 100.0, fame

def total_score(e: Event, dist_mi: Optional[float], comp_0_100: float) -> float:
    prox = proximity_score(dist_mi)
    # Weight "quality/fame" a bit more than distance for concerts.
    if (e.category or "").lower() == "concert":
        return 0.70 * comp_0_100 + 0.30 * prox
    return 0.55 * comp_0_100 + 0.45 * prox

def score_events(events: List[Event], wiki_top_k: int = 200) -> pd.DataFrame:
    """Two-pass scoring:
    1) score without Wikipedia for everyone
    2) compute Wikipedia fame for a prioritized subset to keep runtime reasonable

    Notes:
      - Events without coordinates are kept (distance is blank; proximity_score is 0).
      - Concerts (and especially events tagged 'major_venue') are prioritized for Wikipedia fame.
    """
    rows = []
    for e in events:
        dist = distance_miles(e.lat, e.lon) if (e.lat is not None and e.lon is not None) else None

        comp_base, _ = compelling_score(e, fame_override=0.0, use_wikipedia=False)
        total_base = total_score(e, dist_mi=dist, comp_0_100=comp_base)

        tags = (e.tags or "").lower()
        is_major_venue = "major_venue" in tags
        # Priority for Wikipedia queries (focus on concerts, major venues, prominent halls)
        wiki_priority = 0.0
        if (e.category or "").lower() == "concert":
            wiki_priority += 1.0
        wiki_priority += venue_prominence(e.venue)
        if is_major_venue:
            wiki_priority += 0.9
        if re.search(r"\b(tour|arena|stadium)\b", (e.title or ""), flags=re.I):
            wiki_priority += 0.15

        rows.append({
            "event_obj": e,
            "distance_mi": dist,
            "compelling_base": comp_base,
            "total_base": total_base,
            "has_coords": bool(e.lat is not None and e.lon is not None),
            "wiki_priority": wiki_priority,
            "is_major_venue": is_major_venue,
        })

    # Choose which events get Wikipedia fame computed
    rows_sorted = sorted(
        rows,
        key=lambda r: (r["wiki_priority"], r["has_coords"], r["total_base"], r["compelling_base"]),
        reverse=True
    )
    top_for_wiki = set(id(r["event_obj"]) for r in rows_sorted[:max(0, int(wiki_top_k))])

    enriched = []
    for r in rows_sorted:
        e = r["event_obj"]
        dist = r["distance_mi"]

        fame = 0.0
        if id(e) in top_for_wiki:
            try:
                fame = wikipedia_pageviews_score(guess_primary_entity_for_wiki(e))
            except Exception:
                fame = 0.0

        comp, fame = compelling_score(e, fame_override=fame, use_wikipedia=False)
        total = total_score(e, dist_mi=dist, comp_0_100=comp)

        dist_out = round(float(dist), 1) if isinstance(dist, (int, float)) else None
        prox_out = round(proximity_score(dist_out), 1) if dist_out is not None else None

        is_major_artist = (e.category == "Concert") and (float(fame) >= float(MAJOR_ARTIST_FAME_THRESHOLD))

        enriched.append({
            **event_to_row(e),
            "has_coords": r["has_coords"],
            "distance_mi": dist_out,
            "proximity_score": prox_out,
            "fame_score": round(float(fame), 3),
            "is_major_artist": bool(is_major_artist),
            "compelling_score": round(float(comp), 1),
            "total_score": round(float(total), 1),
        })

    df = pd.DataFrame(enriched)

    # Sort primarily by start time, then score (so weeks read naturally)
    if "start" in df.columns:
        df = df.sort_values(["start", "total_score"], ascending=[True, False])

    return df


In [ ]:
_BASE_TM = "https://app.ticketmaster.com/discovery/v2/events.json"

# Geohash encoder (minimal, enough for Ticketmaster geoPoint)
_BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
def geohash_encode(lat: float, lon: float, precision: int = 7) -> str:
    lat_interval = [-90.0, 90.0]
    lon_interval = [-180.0, 180.0]
    geohash = []
    bit = 0
    ch = 0
    even = True
    bits = [16, 8, 4, 2, 1]
    while len(geohash) < precision:
        if even:
            mid = sum(lon_interval) / 2
            if lon > mid:
                ch |= bits[bit]
                lon_interval[0] = mid
            else:
                lon_interval[1] = mid
        else:
            mid = sum(lat_interval) / 2
            if lat > mid:
                ch |= bits[bit]
                lat_interval[0] = mid
            else:
                lat_interval[1] = mid
        even = not even
        if bit < 4:
            bit += 1
        else:
            geohash.append(_BASE32[ch])
            bit = 0
            ch = 0
    return "".join(geohash)

def _iso_utc(dt: datetime) -> str:
    return dt.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%dT%H:%M:%SZ")

def fetch_ticketmaster_events(start_dt: datetime, end_dt: datetime, radius_miles: int = 45) -> List[Event]:
    if not TICKETMASTER_API_KEY:
        print("⚠️ Ticketmaster API key not set; skipping Ticketmaster source.")
        return []

    geo = geohash_encode(HOME_LAT, HOME_LON, precision=7)
    params = {
        "apikey": TICKETMASTER_API_KEY,
        "geoPoint": geo,
        "radius": radius_miles,
        "unit": "miles",
        "startDateTime": _iso_utc(start_dt),
        "endDateTime": _iso_utc(end_dt),
        "size": 200,
        "sort": "date,asc",
    }

    events: List[Event] = []
    page = 0
    while True:
        params["page"] = page
        r = requests.get(_BASE_TM, params=params, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"Ticketmaster error {r.status_code}: {r.text[:300]}")
        js = r.json()
        items = js.get("_embedded", {}).get("events", [])
        for it in items:
            name = it.get("name")
            url = it.get("url")
            dt_str = it.get("dates", {}).get("start", {}).get("dateTime") or it.get("dates", {}).get("start", {}).get("localDate")
            start = None
            if dt_str:
                try:
                    # Ticketmaster dateTime is usually UTC; treat as such and convert to local tz
                    start = dateparser.isoparse(dt_str)
                    if start.tzinfo is None:
                        start = start.replace(tzinfo=ZoneInfo("UTC"))
                    start = start.astimezone(ZoneInfo(TZ))
                except Exception:
                    pass
            end = start  # many TM events omit end

            venue_name = None
            city = region = address = None
            lat = lon = None
            try:
                v = it.get("_embedded", {}).get("venues", [])[0]
                venue_name = v.get("name")
                city = v.get("city", {}).get("name")
                region = v.get("state", {}).get("stateCode") or v.get("country", {}).get("countryCode")
                address = v.get("address", {}).get("line1")
                loc = v.get("location") or {}
                if loc.get("latitude") and loc.get("longitude"):
                    lat = float(loc["latitude"])
                    lon = float(loc["longitude"])
            except Exception:
                pass

            # Category mapping via segment name when present
            seg = None
            try:
                seg = it.get("classifications", [])[0].get("segment", {}).get("name")
            except Exception:
                pass
            if seg:
                seg_l = seg.lower()
                if "music" in seg_l:
                    cat = "Concert"
                elif "arts" in seg_l or "theatre" in seg_l or "theater" in seg_l:
                    cat = "Play"
                else:
                    cat = "Event"
            else:
                cat = "Event"

            price_min = price_max = None
            if it.get("priceRanges"):
                pr = it["priceRanges"][0]
                price_min = pr.get("min")
                price_max = pr.get("max")

            e = Event(
                title=name,
                category=cat,
                start=start,
                end=end,
                venue=venue_name,
                city=city,
                region=region,
                address=address,
                lat=lat,
                lon=lon,
                source="Ticketmaster",
                url=url,
                description=None,
                price_min=price_min,
                price_max=price_max,
                tags=seg,
            )
            events.append(e)

        # pagination
        page_info = js.get("page", {})
        if not page_info:
            break
        total_pages = page_info.get("totalPages", 0)
        if page + 1 >= total_pages or page >= 4:  # safety cap
            break
        page += 1
    return events


_BASE_TM_VENUES = "https://app.ticketmaster.com/discovery/v2/venues.json"
from difflib import SequenceMatcher

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

def _best_name_match_score(target: str, candidate: str) -> float:
    t = _norm(target)
    c = _norm(candidate)
    if not t or not c:
        return 0.0
    if t == c:
        return 1.0
    if t in c or c in t:
        return 0.85
    return SequenceMatcher(None, t, c).ratio()

def resolve_ticketmaster_venue_id(spec: Dict[str, Any]) -> Optional[str]:
    """Resolve a Ticketmaster venueId for a venue spec (keyword + optional city/state).
    If spec includes a literal venueId, it will be used directly.
    Uses a small disk cache to reduce API calls.
    """
    # Allow manual override
    if spec.get("venueId"):
        return str(spec.get("venueId"))
    if not TICKETMASTER_API_KEY:
        return None

    keyword = spec.get("keyword") or spec.get("label") or ""
    city = spec.get("city")
    state = spec.get("stateCode")
    cache_key = f"tm:venueId:{_norm(keyword)}:{_norm(city or '')}:{_norm(state or '')}"
    if cache_key in cache:
        vid = cache[cache_key]
        return vid or None

    params = {
        "apikey": TICKETMASTER_API_KEY,
        "keyword": keyword,
        "size": 10,
        "countryCode": "US",
    }
    if city:
        params["city"] = city
    if state:
        params["stateCode"] = state

    r = requests.get(_BASE_TM_VENUES, params=params, timeout=20, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    if r.status_code != 200:
        cache[cache_key] = ""
        return None
    js = r.json()
    venues = js.get("_embedded", {}).get("venues", []) or []

    # If we over-constrained with city/state (some TM records use different city labels),
    # retry once with keyword-only.
    if not venues and (city or state):
        params2 = {
            "apikey": TICKETMASTER_API_KEY,
            "keyword": keyword,
            "size": 10,
            "countryCode": "US",
        }
        r2 = requests.get(_BASE_TM_VENUES, params=params2, timeout=20, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
        if r2.status_code == 200:
            venues = (r2.json().get("_embedded", {}) or {}).get("venues", []) or []

    if not venues:
        cache[cache_key] = ""
        return None

    # pick best match
    want = spec.get("label") or keyword
    aliases = spec.get("aliases") or []
    best = None
    best_score = -1.0
    for v in venues:
        name = v.get("name", "")
        s = _best_name_match_score(want, name)
        # alias boost
        for a in aliases:
            s = max(s, _best_name_match_score(a, name) * 0.98)
        # city boost
        v_city = (v.get("city", {}) or {}).get("name", "")
        if city and _norm(city) in _norm(v_city):
            s += 0.08
        if s > best_score:
            best_score = s
            best = v

    vid = (best or {}).get("id") if best_score >= 0.55 else None
    cache[cache_key] = vid or ""
    return vid

def fetch_ticketmaster_music_events_for_venue(venue_id: str, start_dt: datetime, end_dt: datetime) -> List[Event]:
    if not TICKETMASTER_API_KEY:
        return []
    params = {
        "apikey": TICKETMASTER_API_KEY,
        "venueId": venue_id,
        "classificationName": "music",
        "startDateTime": _iso_utc(start_dt),
        "endDateTime": _iso_utc(end_dt),
        "size": 200,
        "sort": "date,asc",
    }
    r = requests.get(_BASE_TM, params=params, timeout=25, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    if r.status_code != 200:
        return []
    js = r.json()
    items = js.get("_embedded", {}).get("events", []) or []
    out: List[Event] = []
    for it in items:
        name = it.get("name")
        url = it.get("url")
        dt_str = it.get("dates", {}).get("start", {}).get("dateTime") or it.get("dates", {}).get("start", {}).get("localDate")
        start = None
        if dt_str:
            try:
                start = dateparser.isoparse(dt_str)
                if start.tzinfo is None:
                    start = start.replace(tzinfo=ZoneInfo("UTC"))
                start = start.astimezone(ZoneInfo(TZ))
            except Exception:
                start = None
        end = start

        venue_name = None
        city = region = address = None
        lat = lon = None
        try:
            v = it.get("_embedded", {}).get("venues", [])[0]
            venue_name = v.get("name")
            city = v.get("city", {}).get("name")
            region = v.get("state", {}).get("stateCode") or v.get("country", {}).get("countryCode")
            address = v.get("address", {}).get("line1")
            loc = v.get("location") or {}
            lat = float(loc.get("latitude")) if loc.get("latitude") else None
            lon = float(loc.get("longitude")) if loc.get("longitude") else None
        except Exception:
            pass

        price_min = price_max = None
        if it.get("priceRanges"):
            try:
                pr = it["priceRanges"][0]
                price_min = pr.get("min")
                price_max = pr.get("max")
            except Exception:
                pass

        e = Event(
            title=name or "(untitled)",
            category="Concert",
            start=start,
            end=end,
            venue=venue_name,
            city=city,
            region=region,
            address=address,
            lat=lat,
            lon=lon,
            source="Ticketmaster",
            url=url,
            description=None,
            price_min=price_min,
            price_max=price_max,
            tags="major_venue",
        )
        out.append(e)
    return out

def fetch_ticketmaster_major_venue_concerts(start_dt: datetime, end_dt: datetime, venues: List[Dict[str, Any]] = None) -> List[Event]:
    """Fetch concerts for a curated set of major venues (LA / OC / SD) by resolving venueId then querying events by venueId."""
    if not TICKETMASTER_API_KEY:
        print("⚠️ Ticketmaster API key not set; skipping major-venue pass.")
        return []
    venues = venues or MAJOR_CONCERT_VENUES
    out: List[Event] = []
    for spec in venues:
        vid = resolve_ticketmaster_venue_id(spec)
        if not vid:
            continue
        out += fetch_ticketmaster_music_events_for_venue(vid, start_dt, end_dt)
        time.sleep(0.15)  # polite pacing
    return out


In [ ]:
PACIFIC_SYMPHONY_URL = "https://www.pacificsymphony.org/get-tickets"

def scrape_pacific_symphony(start_dt: datetime, end_dt: datetime) -> List[Event]:
    events: List[Event] = []
    r = requests.get(PACIFIC_SYMPHONY_URL, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    for h3 in soup.find_all(["h3", "h2"]):
        title = h3.get_text(" ", strip=True)
        if not title or len(title) < 3:
            continue
        # Heuristic: pacific symphony listing titles are in h3 with nearby li containing date range
        li = h3.find_next("li")
        if not li:
            continue
        dt_text = li.get_text(" ", strip=True)
        if "|" not in dt_text and "AM" not in dt_text and "PM" not in dt_text:
            continue

        # Venue is usually next anchor after li
        venue_a = li.find_next("a")
        venue = venue_a.get_text(" ", strip=True) if venue_a else None

        # Read more link
        read_more = None
        for a in h3.find_all_next("a", limit=10):
            t = a.get_text(" ", strip=True).lower()
            if t in {"read more", "details", "more info"}:
                read_more = a.get("href")
                break

        # Parse date span + time
        parts = [p.strip() for p in dt_text.split("|", 1)]
        date_part = parts[0].strip("• ").strip()
        time_part = parts[1].strip() if len(parts) > 1 else ""
        sd, ed = parse_date_span(date_part)
        tm = None
        if time_part:
            tm = parse_time_text(time_part.split(",")[0].strip())
        start = end = None
        if sd:
            start = datetime.combine(sd, tm or dtime(19, 30), ZoneInfo(TZ))
        if ed:
            end = datetime.combine(ed, tm or dtime(19, 30), ZoneInfo(TZ))

        e = Event(
            title=title,
            category="Concert",
            start=start,
            end=end,
            venue=venue,
            city="Costa Mesa" if venue and "Segerstrom" in venue else None,
            region="CA",
            address=None,
            lat=None,
            lon=None,
            source="PacificSymphony",
            url=read_more if (read_more and read_more.startswith("http")) else (f"https://www.pacificsymphony.org{read_more}" if read_more else None),
            description=None,
        )
        events.append(e)

    # Deduplicate by (title, start_date)
    dedup = {}
    for e in events:
        key = (e.title, e.start.date() if e.start else None)
        dedup[key] = e
    return list(dedup.values())


In [ ]:
LAGUNA_SHOWS_URL = "https://lagunaplayhouse.com/tickets-and-events/shows/"

def scrape_laguna_playhouse(start_dt: datetime, end_dt: datetime) -> List[Event]:
    r = requests.get(LAGUNA_SHOWS_URL, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    events: List[Event] = []
    for h2 in soup.find_all("h2"):
        title = h2.get_text(" ", strip=True)
        if not title or title.lower() in {"shows", "2025 - 2026 productions"}:
            continue

        # Find the first date-looking text after this h2
        date_text = None
        for el in h2.find_all_next(["p", "div", "span"], limit=6):
            txt = el.get_text(" ", strip=True)
            if re.search(r"\b(20\d{2}|19\d{2})\b", txt) or re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\b", txt, re.I):
                sd, ed = parse_date_span(txt)
                if sd:
                    date_text = txt
                    break

        if not date_text:
            continue

        sd, ed = parse_date_span(date_text)
        start = datetime.combine(sd, dtime(19, 30), ZoneInfo(TZ)) if sd else None
        end = datetime.combine(ed, dtime(22, 0), ZoneInfo(TZ)) if ed else start

        # Buy Tickets link
        buy = None
        for a in h2.find_all_next("a", limit=8):
            if "buy" in a.get_text(" ", strip=True).lower() and "ticket" in a.get_text(" ", strip=True).lower():
                buy = a.get("href")
                break

        e = Event(
            title=title,
            category="Play",
            start=start,
            end=end,
            venue="Laguna Playhouse",
            city="Laguna Beach",
            region="CA",
            address="606 Laguna Canyon Rd, Laguna Beach, CA 92651",
            lat=None,
            lon=None,
            source="LagunaPlayhouse",
            url=buy if (buy and buy.startswith("http")) else (f"https://lagunaplayhouse.com{buy}" if buy else None),
            description=f"Run dates: {date_text}",
        )
        events.append(e)

    # Deduplicate
    dedup = {}
    for e in events:
        dedup[(e.title, e.start.date() if e.start else None)] = e
    return list(dedup.values())


In [ ]:
SCR_SEASON_URL = "https://www.scr.org/plays/plays-landing/2025-26-season-at-a-glance/"

def _absolute(url: str) -> str:
    if not url:
        return url
    if url.startswith("http"):
        return url
    return f"https://www.scr.org{url}"

def extract_scr_production_links() -> List[str]:
    r = requests.get(SCR_SEASON_URL, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/plays/productions/" in href:
            links.add(_absolute(href.split("#")[0]))
    return sorted(links)

def scrape_scr_production(url: str) -> Optional[Event]:
    r = requests.get(url, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    # Title from og:title
    title = None
    og = soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        title = og["content"].split("|")[0].strip()
    if not title:
        # fallback to largest heading
        h = soup.find(["h1", "h2"])
        title = h.get_text(" ", strip=True) if h else url

    text = soup.get_text("\n", strip=True)

    # Stage
    stage = None
    for s in ["Segerstrom Stage", "Julianne Argyros Stage", "Nicholas Studio"]:
        if s in text:
            stage = s
            break

    # Age recommendation
    age_hint = None
    m = re.search(r"(Recommendation|Recommended)\s*:?\s*Age\s*(\d{1,2})\s*\+?", text, flags=re.I)
    if m:
        age_hint = f"{m.group(2)}+"
    else:
        m2 = re.search(r"Recommended\s+for\s+ages\s+(\d{1,2})\s*\+?", text, flags=re.I)
        if m2:
            age_hint = f"{m2.group(1)}+"

    # Dates: prefer "Regular Performances:"
    date_span = None
    m = re.search(r"Regular Performances:\s*([^\n]+)", text, flags=re.I)
    if m:
        date_span = m.group(1).strip()
    else:
        # fallback: look for a line containing a year and a month
        for line in text.split("\n"):
            if re.search(r"\b(20\d{2}|19\d{2})\b", line) and re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\b", line, re.I):
                # avoid "Tickets on Sale" etc
                if "tickets on sale" in line.lower():
                    continue
                date_span = line.strip()
                break

    sd, ed = parse_date_span(date_span or "")
    if not sd:
        return None
    start = datetime.combine(sd, dtime(19, 30), ZoneInfo(TZ))
    end = datetime.combine(ed or sd, dtime(22, 0), ZoneInfo(TZ))

    desc = None
    # short description: first paragraph-ish
    # (keep simple; full text is huge)
    for line in text.split("\n"):
        if len(line) > 80 and "On " not in line[:4]:
            desc = line[:300]
            break

    e = Event(
        title=title,
        category="Play",
        start=start,
        end=end,
        venue="South Coast Repertory" if not stage else stage,
        city="Costa Mesa",
        region="CA",
        address="655 Town Center Dr, Costa Mesa, CA 92628",
        lat=None,
        lon=None,
        source="SCR",
        url=url,
        description=(desc + (f" (Recommended age: {age_hint})" if age_hint else "")) if desc else None,
        age_hint=age_hint,
    )
    return e

def scrape_scr(start_dt: datetime, end_dt: datetime, max_productions: int = 10) -> List[Event]:
    links = extract_scr_production_links()
    # Scrape a limited number (season pages can be big)
    out: List[Event] = []
    for url in links[:max_productions]:
        try:
            ev = scrape_scr_production(url)
            if ev:
                out.append(ev)
        except Exception as ex:
            print(f"SCR scrape failed for {url}: {ex}")
            continue
    return out


In [ ]:
LIVENATION_HOB_URL = "https://www.livenation.com/venue/KovZpZAEA67A/house-of-blues-anaheim-events"

def _parse_livenation_datetime(line: str, tz: str = TZ) -> Optional[datetime]:
    # Example: "Fri Dec 19, 2025 ▪︎ 6:30PM"
    s = line.replace("▪︎", " ").replace("•", " ")
    s = re.sub(r"\s+", " ", s).strip()
    try:
        dt = dateparser.parse(s, fuzzy=True)
        if dt is None:
            return None
        # LiveNation listings are local
        return dt.replace(tzinfo=ZoneInfo(tz)) if dt.tzinfo is None else dt.astimezone(ZoneInfo(tz))
    except Exception:
        return None

def scrape_livenation_house_of_blues(start_dt: datetime, end_dt: datetime) -> List[Event]:
    r = requests.get(LIVENATION_HOB_URL, timeout=30, headers={"User-Agent": "soc-weekly-event-finder/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    text = soup.get_text("\n", strip=True)

    # Start scanning after "Advertisement" if present
    if "Advertisement" in text:
        text = text.split("Advertisement", 1)[1]

    chunks = text.split("Buy Tickets")
    events: List[Event] = []
    for chunk in chunks[:-1]:
        lines = [ln.strip() for ln in chunk.split("\n") if ln.strip()]
        # remove obvious junk
        lines = [ln for ln in lines if not ln.lower().startswith("image:") and ln.lower() not in {"more info", "today", "tomorrow"}]
        if len(lines) < 4:
            continue

        # Find a line with a weekday + time (best signal)
        dt_line = None
        dt_val = None
        for ln in lines:
            if "am" in ln.lower() or "pm" in ln.lower():
                if re.search(r"\b(mon|tue|wed|thu|fri|sat|sun)\b", ln.lower()):
                    dt_line = ln
                    dt_val = _parse_livenation_datetime(ln)
                    break
        if not dt_val:
            continue

        # Title: pick the most title-like line near dt_line (often right before venue line)
        # We'll take the nearest non-date line above dt_line that isn't the venue
        idx = lines.index(dt_line)
        venue = None
        # Look around for venue-like line
        for ln in lines[max(0, idx-4): idx+3]:
            if "house of blues" in ln.lower():
                venue = ln
                break
        # Candidate title lines: around idx-6..idx
        candidates = []
        for ln in lines[max(0, idx-6): idx]:
            if any(w in ln.lower() for w in ["house of blues", "the parish", "anaheim"]):
                continue
            if re.search(r"\b\d{4}\b", ln) and re.search(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b", ln.lower()):
                continue
            if len(ln) < 3 or len(ln) > 120:
                continue
            candidates.append(ln)
        title = candidates[-1] if candidates else None
        if not title:
            continue

        # Event URL: try to find a ticketmaster link nearby
        url = None
        for a in soup.find_all("a", href=True):
            if "ticketmaster.com" in a["href"] and title.lower() in a.get_text(" ", strip=True).lower():
                url = a["href"]
                break

        e = Event(
            title=title,
            category="Concert",
            start=dt_val,
            end=dt_val,
            venue=venue or "House of Blues Anaheim",
            city="Anaheim",
            region="CA",
            address="400 W Disney Way #337, Anaheim, CA 92802",
            source="LiveNation",
            url=url,
        )
        events.append(e)

    # Deduplicate
    dedup = {}
    for e in events:
        key = (e.title, e.start)
        dedup[key] = e
    return list(dedup.values())


In [ ]:
def hydrate_coords(events: List[Event]) -> List[Event]:
    for e in events:
        if e.lat is not None and e.lon is not None:
            continue
        lat, lon = infer_coords(e.venue, e.address)
        if lat is None and e.address:
            try:
                lat, lon = geocode_address(e.address)
            except Exception:
                lat, lon = (None, None)
        e.lat, e.lon = lat, lon
    return events

def filter_events(events: List[Event], window_start: datetime, window_end: datetime) -> List[Event]:
    out: List[Event] = []
    for e in events:
        if not e.title or e.start is None:
            continue

        # Hard venue blacklist (e.g., Hamburger Mary's)
        hay = " ".join([str(e.venue or ""), str(e.address or "")]).lower()
        if any(sub.lower() in hay for sub in (EXCLUDED_VENUE_SUBSTRINGS or [])):
            continue


        st = e.start
        en = e.end

        # Normalize to tz-aware datetimes for consistent window comparisons
        if getattr(st, "tzinfo", None) is None:
            st = st.replace(tzinfo=ZoneInfo(TZ))
        if en is not None and getattr(en, "tzinfo", None) is None:
            en = en.replace(tzinfo=ZoneInfo(TZ))

        # Keep only events whose START occurs within the target window.
        # (This avoids long-running listings that overlap the window but aren't specific to this week.)
        if not (window_start <= st <= window_end):
            continue

        e.start, e.end = st, en

        if is_kids_event(e):
            continue
        if is_promo_event(e):
            continue
        # Nightlife / low-signal filter: aggressively exclude
        if is_nightlife_event(e) and venue_prominence(e.venue) < 0.85:
            continue
        if is_low_signal(e):
            continue

        out.append(e)
    return out

def dedupe(events: List[Event]) -> List[Event]:
    priority = {"Ticketmaster": 3, "PacificSymphony": 2, "SCR": 2, "LagunaPlayhouse": 2, "LiveNation": 1}
    best = {}
    for e in events:
        k = (re.sub(r"\s+", " ", e.title.strip().lower()), e.start.date() if e.start else None)
        p = priority.get(e.source or "", 0)
        if k not in best or p > priority.get(best[k].source or "", 0):
            best[k] = e
    return list(best.values())

# --- Collect ---
events_all: List[Event] = []

try:
    events_all += fetch_ticketmaster_events(START_DT, END_DT, radius_miles=RADIUS_MILES)
except Exception as ex:
    print(f"Ticketmaster fetch failed: {ex}")

try:
    if INCLUDE_MAJOR_VENUE_CONCERTS:
        events_all += fetch_ticketmaster_major_venue_concerts(START_DT, END_DT)
except Exception as ex:
    print(f"Ticketmaster major-venue fetch failed: {ex}")

try:
    events_all += scrape_pacific_symphony(START_DT, END_DT)
except Exception as ex:
    print(f"Pacific Symphony scrape failed: {ex}")

try:
    events_all += scrape_laguna_playhouse(START_DT, END_DT)
except Exception as ex:
    print(f"Laguna Playhouse scrape failed: {ex}")

try:
    events_all += scrape_scr(START_DT, END_DT, max_productions=12)
except Exception as ex:
    print(f"SCR scrape failed: {ex}")

if not TICKETMASTER_API_KEY:
    try:
        events_all += scrape_livenation_house_of_blues(START_DT, END_DT)
    except Exception as ex:
        print(f"LiveNation scrape failed: {ex}")

print(f"Collected raw events: {len(events_all)}")

events_all = hydrate_coords(events_all)
events_all = filter_events(events_all, START_DT, END_DT)
events_all = dedupe(events_all)

print(f"After filtering+dedupe: {len(events_all)}")

df = score_events(events_all, wiki_top_k=250)
df.head()


In [ ]:
def fmt_price(row) -> str:
    if pd.isna(row.get("price_min")) and pd.isna(row.get("price_max")):
        return ""
    lo = row.get("price_min")
    hi = row.get("price_max")
    if pd.notna(lo) and pd.notna(hi) and lo != hi:
        return f"${lo:.0f}–${hi:.0f}"
    if pd.notna(lo):
        return f"from ${lo:.0f}"
    if pd.notna(hi):
        return f"up to ${hi:.0f}"
    return ""

def render_section(df_section: pd.DataFrame, title: str):
    if df_section.empty:
        print(f"\n### {title}\n(no items in top list)")
        return
    display_cols = [
        "title", "start", "venue", "distance_mi", "total_score", "compelling_score", "source", "url"
    ]
    d = df_section[display_cols].copy()
    d["start"] = d["start"].apply(lambda x: x.strftime('%a %b %d, %I:%M %p') if pd.notna(x) else "")
    display(d)

if df.empty:
    print("No events found after filtering. Try increasing radius, adding Ticketmaster API key, or loosening filters.")
else:
    top = df.copy()

    # Sectioned output
    print("## Top picks (sectioned)")
    for cat, title in [("Concert", "Concerts"), ("Play", "Plays / Theatre"), ("Event", "Other Events")]:
        render_section(top[top["category"] == cat], title)

    # Also show the full top list in one table
    print("\n## Top list (all categories)")
    display_cols = ["category", "title", "start", "venue", "distance_mi", "total_score", "compelling_score", "source", "url"]
    d = top[display_cols].copy()
    d["start"] = d["start"].apply(lambda x: x.strftime('%a %b %d, %I:%M %p') if pd.notna(x) else "")
    display(d)


## 4) (Optional) Export a shareable HTML report

In [ ]:
from pathlib import Path
from html import escape

def _fmt_clock(dt: datetime) -> str:
    return dt.strftime("%I:%M %p").lstrip("0")

def _ensure_tz(dt: Optional[datetime]) -> Optional[datetime]:
    if dt is None:
        return None
    if getattr(dt, "tzinfo", None) is None:
        return dt.replace(tzinfo=ZoneInfo(TZ))
    return dt

def _fmt_when(start: Optional[datetime], end: Optional[datetime]) -> str:
    start = _ensure_tz(start)
    end = _ensure_tz(end)

    if start is None:
        return "TBA"

    s_local = start.astimezone(ZoneInfo(TZ))
    if end is None:
        return f"{s_local:%a %b %d} • {_fmt_clock(s_local)}"

    e_local = end.astimezone(ZoneInfo(TZ))
    if s_local.date() == e_local.date():
        return f"{s_local:%a %b %d} • {_fmt_clock(s_local)}–{_fmt_clock(e_local)}"
    return f"{s_local:%a %b %d} • {_fmt_clock(s_local)} → {e_local:%a %b %d} • {_fmt_clock(e_local)}"

def _repo_root() -> Path:
    cwd = Path.cwd()
    # If running from notebooks/ in GitHub Actions
    if cwd.name == "notebooks" and (cwd.parent / ".github").is_dir():
        return cwd.parent
    if (cwd / ".github").is_dir():
        return cwd
    if (cwd.parent / ".github").is_dir():
        return cwd.parent
    return cwd

def _week_ranges(window_start: datetime, window_end: datetime) -> List[Tuple[date, date]]:
    start_d = window_start.astimezone(ZoneInfo(TZ)).date()
    end_d = window_end.astimezone(ZoneInfo(TZ)).date()
    total_days = (end_d - start_d).days + 1
    weeks = max(1, math.ceil(total_days / 7))
    ranges = []
    for i in range(weeks):
        ws = start_d + timedelta(days=7*i)
        we = min(end_d, ws + timedelta(days=6))
        ranges.append((ws, we))
    return ranges

def make_html_report(df_all: pd.DataFrame, window_start: datetime, window_end: datetime) -> str:
    ws_label = window_start.strftime("%a %b %d, %Y")
    we_label = window_end.strftime("%a %b %d, %Y")
    generated = datetime.now(ZoneInfo(TZ)).strftime("%a %b %d, %Y • %I:%M %p %Z").replace(" 0", " ")

    # Normalize datetimes for consistent grouping
    df = df_all.copy()
    if not df.empty and "start" in df.columns:
        df["start"] = df["start"].apply(_ensure_tz)

    ranges = _week_ranges(window_start, window_end)

    css = """
    :root{
      --bg:#0b1220;
      --panel:#10182a;
      --card:#121a2b;
      --muted:#9fb0d0;
      --text:#e7eefc;
      --accent:#7aa2ff;
      --accent2:#5eead4;
      --border:rgba(255,255,255,.10);
      --shadow: 0 10px 28px rgba(0,0,0,.25);
      --radius: 18px;
      --radius2: 14px;
    }
    *{box-sizing:border-box}
    html,body{height:100%}
    body{
      margin:0;
      background: radial-gradient(1200px 600px at 20% 0%, rgba(122,162,255,.22), transparent 55%),
                  radial-gradient(900px 500px at 90% 10%, rgba(94,234,212,.14), transparent 45%),
                  var(--bg);
      color:var(--text);
      font-family: Inter, ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, "Apple Color Emoji","Segoe UI Emoji";
      line-height:1.45;
    }
    a{color:inherit}
    .wrap{max-width:1120px; margin:0 auto; padding:28px 18px 50px;}
    header{
      padding:22px 20px;
      background: linear-gradient(180deg, rgba(255,255,255,.08), rgba(255,255,255,.03));
      border:1px solid var(--border);
      border-radius: var(--radius);
      box-shadow: var(--shadow);
      overflow:hidden;
      position:relative;
    }
    header:before{
      content:""; position:absolute; inset:-2px;
      background: radial-gradient(700px 220px at 0% 0%, rgba(122,162,255,.25), transparent 60%),
                  radial-gradient(600px 220px at 100% 0%, rgba(94,234,212,.16), transparent 60%);
      opacity:.85; pointer-events:none;
    }
    header > *{position:relative}
    .kicker{color:var(--muted); font-size:14px; margin:0 0 6px}
    h1{margin:0 0 8px; font-size:30px; letter-spacing:-.02em}
    .meta{display:flex; flex-wrap:wrap; gap:10px; margin-top:10px; color:var(--muted); font-size:13px}
    .pill{
      display:inline-flex; align-items:center; gap:8px;
      padding:6px 10px;
      border:1px solid var(--border);
      border-radius:999px;
      background: rgba(255,255,255,.03);
      backdrop-filter: blur(6px);
    }
    .dot{width:8px; height:8px; border-radius:999px; background: var(--accent2); display:inline-block}
    .nav{
      display:flex; flex-wrap:wrap; gap:10px;
      margin:16px 0 10px;
    }
    .nav a{
      text-decoration:none;
      display:inline-flex; align-items:center;
      padding:8px 12px;
      border-radius:999px;
      border:1px solid var(--border);
      background: rgba(255,255,255,.03);
      color:var(--text);
      font-size:13px;
    }
    .nav a:hover{border-color: rgba(122,162,255,.55)}
    .controls{
      display:flex; flex-wrap:wrap; gap:10px;
      margin:14px 0 0;
      align-items:center;
    }
    .search{
      flex:1 1 280px;
      min-width:220px;
      display:flex;
      align-items:center;
      gap:10px;
      padding:10px 12px;
      border-radius: 14px;
      border:1px solid var(--border);
      background: rgba(255,255,255,.03);
    }
    .search input{
      width:100%;
      border:none;
      outline:none;
      background:transparent;
      color:var(--text);
      font-size:14px;
      font-family: inherit;
    }
    .btn{
      cursor:pointer;
      border:1px solid var(--border);
      background: rgba(255,255,255,.03);
      color: var(--text);
      border-radius: 14px;
      padding:10px 12px;
      font-size:13px;
      font-family: inherit;
    }
    .btn:hover{border-color: rgba(122,162,255,.55)}
    details{
      border:1px solid var(--border);
      border-radius: var(--radius);
      background: rgba(255,255,255,.03);
      box-shadow: var(--shadow);
      overflow:hidden;
    }
    details.week{margin-top:16px}
    details.cat{margin-top:10px; border-radius: var(--radius2)}
    summary{
      list-style:none;
      cursor:pointer;
      user-select:none;
      padding:14px 16px;
      display:flex;
      align-items:center;
      justify-content:space-between;
      gap:12px;
    }
    summary::-webkit-details-marker{display:none}
    .sum-left{display:flex; align-items:baseline; gap:10px; flex-wrap:wrap}
    .sum-title{font-weight:700; font-size:16px}
    .sum-sub{color:var(--muted); font-size:13px}
    .count{
      color: var(--muted);
      font-size:13px;
      border:1px solid var(--border);
      border-radius:999px;
      padding:6px 10px;
      background: rgba(0,0,0,.12);
      white-space:nowrap;
    }
    .content{padding:0 16px 16px}
    .grid{
      display:grid;
      grid-template-columns: repeat(3, minmax(0, 1fr));
      gap:12px;
      margin-top:12px;
    }
    @media(max-width: 980px){
      .grid{grid-template-columns: repeat(2, minmax(0, 1fr));}
    }
    @media(max-width: 640px){
      .grid{grid-template-columns: 1fr;}
      h1{font-size:26px}
    }
    .card{
      background: linear-gradient(180deg, rgba(255,255,255,.07), rgba(255,255,255,.03));
      border:1px solid var(--border);
      border-radius: var(--radius2);
      padding:14px;
      position:relative;
      overflow:hidden;
    }
    .card:before{
      content:""; position:absolute; inset:-2px;
      background: radial-gradient(500px 220px at 0% 0%, rgba(122,162,255,.12), transparent 60%);
      opacity:.65; pointer-events:none;
    }
    .card > *{position:relative}
    .title{font-weight:750; font-size:15px; margin:0 0 10px}
    .title a{text-decoration:none}
    .title a:hover{text-decoration:underline}
    .row{display:flex; flex-wrap:wrap; gap:8px}
    .chip{
      border:1px solid var(--border);
      background: rgba(0,0,0,.12);
      border-radius:999px;
      padding:6px 10px;
      font-size:12px;
      color:var(--muted);
      white-space:nowrap;
    }
    .chip strong{color:var(--text); font-weight:650}
    .badges{display:flex; flex-wrap:wrap; gap:8px; margin-bottom:10px}
    .badge{
      font-size:12px;
      padding:6px 10px;
      border-radius:999px;
      border:1px solid var(--border);
      background: rgba(122,162,255,.10);
      color: var(--text);
    }
    .badge.major{background: rgba(94,234,212,.10)}
    .badge.dim{background: rgba(255,255,255,.04); color: var(--muted)}
    .desc{color:var(--muted); font-size:13px; margin-top:10px}
    .hidden{display:none !important}
    footer{color:var(--muted); font-size:12px; margin-top:18px; text-align:center}
    """

    def build_cards(d: pd.DataFrame) -> str:
        cards = []
        for _, r in d.iterrows():
            title = escape(str(r.get("title", "(untitled)")))
            url = str(r.get("url") or "").strip()
            link = f'<a href="{escape(url)}" target="_blank" rel="noopener noreferrer">{title}</a>' if url else title

            when = _fmt_when(r.get("start"), r.get("end"))
            venue = escape(str(r.get("venue") or ""))
            city = escape(str(r.get("city") or ""))
            where = ", ".join([x for x in [venue, city] if x]).strip() or "—"

            dist = r.get("distance_mi")
            dist_s = f"{dist:.1f} mi" if isinstance(dist, (int, float)) else "—"

            total = r.get("total_score")
            total_s = f"{total:.1f}" if isinstance(total, (int, float)) else "—"

            comp = r.get("compelling_score")
            comp_s = f"{comp:.1f}" if isinstance(comp, (int, float)) else "—"

            fame = r.get("fame_score")
            fame_s = "—"
            if isinstance(fame, (int, float)) and float(fame) > 0:
                fame_s = f"{int(round(float(fame)*100))}%"

            price = fmt_price(r)
            src = escape(str(r.get("source") or ""))

            desc = str(r.get("description") or "").strip()
            desc = re.sub(r"\s+", " ", desc)
            if len(desc) > 220:
                desc = desc[:217].rstrip() + "…"
            desc_html = f'<div class="desc">{escape(desc)}</div>' if desc else ""

            tags = (str(r.get("tags") or "") or "").lower()
            is_major_venue = "major_venue" in tags
            is_major_artist = bool(r.get("is_major_artist"))

            badges = []
            if is_major_artist:
                badges.append('<span class="badge major">Major Artist</span>')
            if is_major_venue:
                badges.append('<span class="badge">Major Venue</span>')
            if not bool(r.get("has_coords")):
                badges.append('<span class="badge dim">No map coords</span>')

            badge_html = f'<div class="badges">{"".join(badges)}</div>' if badges else ""

            chips = [
                f'<span class="chip"><strong>When</strong> {escape(when)}</span>',
                f'<span class="chip"><strong>Where</strong> {where}</span>',
            ]
            if dist_s != "—":
                chips.append(f'<span class="chip"><strong>Distance</strong> {dist_s}</span>')
            chips.append(f'<span class="chip"><strong>Score</strong> {total_s} <span style="color:var(--muted)">(comp {comp_s})</span></span>')
            if (str(r.get("category") or "").lower() == "concert") and fame_s != "—":
                chips.append(f'<span class="chip"><strong>Fame</strong> {fame_s}</span>')
            if price:
                chips.append(f'<span class="chip"><strong>Price</strong> {escape(price)}</span>')
            if src:
                chips.append(f'<span class="chip"><strong>Source</strong> {src}</span>')

            # searchable text blob
            search_blob = " ".join([str(r.get("category") or ""), str(r.get("title") or ""), str(r.get("venue") or ""), str(r.get("city") or "")]).lower()

            cards.append(f"""<article class="card" data-search="{escape(search_blob)}">
                {badge_html}
                <div class="title">{link}</div>
                <div class="row">{"".join(chips)}</div>
                {desc_html}
              </article>""")
        return "".join(cards)

    def week_block(week_idx: int, ws: date, we: date) -> str:
        # Filter df to this week range
        if df.empty:
            dweek = df
        else:
            def _in_week(x):
                x = _ensure_tz(x)
                if x is None:
                    return False
                xd = x.astimezone(ZoneInfo(TZ)).date()
                return ws <= xd <= we
            dweek = df[df["start"].apply(_in_week)].copy()

        # Subsections
        major = dweek[(dweek.get("is_major_artist", False) == True)].copy()
        # Avoid duplicates: exclude major artists from the general concert list
        concerts = dweek[(dweek["category"] == "Concert") & (dweek.get("is_major_artist", False) != True)].copy()
        plays = dweek[dweek["category"] == "Play"].copy()
        other = dweek[dweek["category"] == "Event"].copy()

        # Sorting
        if not major.empty:
            major = major.sort_values(["fame_score", "total_score", "start"], ascending=[False, False, True])
        if not concerts.empty:
            concerts = concerts.sort_values(["total_score", "fame_score", "start"], ascending=[False, False, True])
        if not plays.empty:
            plays = plays.sort_values(["total_score", "start"], ascending=[False, True])
        if not other.empty:
            other = other.sort_values(["total_score", "start"], ascending=[False, True])

        week_id = f"week-{week_idx}"
        open_attr = " open" if week_idx == 1 else ""
        sum_label = f"Week {week_idx}"
        range_label = f"{ws:%b %d}–{we:%b %d}"
        total_count = len(dweek)

        def cat_details(title: str, dcat: pd.DataFrame, open_default: bool = False) -> str:
            cnt = len(dcat)
            od = " open" if open_default and cnt > 0 else ""
            content = '<p class="sum-sub" style="margin:10px 0 0">(none)</p>' if cnt == 0 else f'<div class="grid">{build_cards(dcat)}</div>'
            return f"""<details class="cat"{od}>
              <summary>
                <div class="sum-left">
                  <span class="sum-title">{escape(title)}</span>
                  <span class="sum-sub">{cnt} item{'s' if cnt != 1 else ''}</span>
                </div>
                <span class="count">{cnt}</span>
              </summary>
              <div class="content">{content}</div>
            </details>"""

        return f"""<details class="week" id="{week_id}"{open_attr}>
          <summary>
            <div class="sum-left">
              <span class="sum-title">{escape(sum_label)}</span>
              <span class="sum-sub">{escape(range_label)}</span>
            </div>
            <span class="count">{total_count} event{'s' if total_count != 1 else ''}</span>
          </summary>
          <div class="content">
            {cat_details('Major Artists', major, open_default=True)}
            {cat_details('Concerts', concerts, open_default=False)}
            {cat_details('Plays / Theatre', plays, open_default=False)}
            {cat_details('Other Events', other, open_default=False)}
          </div>
        </details>"""

    week_nav = []
    week_sections = []
    for i, (ws_d, we_d) in enumerate(ranges, start=1):
        week_id = f"week-{i}"
        week_nav.append(f'<a href="#{week_id}">Week {i}: {ws_d:%b %d}–{we_d:%b %d}</a>')
        week_sections.append(week_block(i, ws_d, we_d))

    js = """
    (function(){
      const q = document.getElementById('q');
      const cards = () => Array.from(document.querySelectorAll('.card'));
      q.addEventListener('input', () => {
        const term = (q.value || '').trim().toLowerCase();
        cards().forEach(c => {
          const hay = (c.getAttribute('data-search') || '');
          c.classList.toggle('hidden', term && !hay.includes(term));
        });
      });

      function setAllDetails(open){
        document.querySelectorAll('details.week, details.cat').forEach(d => d.open = open);
      }
      document.getElementById('expandAll').addEventListener('click', () => setAllDetails(true));
      document.getElementById('collapseAll').addEventListener('click', () => setAllDetails(false));
    })();
    """

    total_events = int(len(df)) if isinstance(df, pd.DataFrame) else 0
    major_events = int(df.get("is_major_artist", pd.Series([])).sum()) if (isinstance(df, pd.DataFrame) and "is_major_artist" in df.columns) else 0

    html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Southern OC — 4 Week Event Finder</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap" rel="stylesheet">
  <style>{css}</style>
</head>
<body>
  <div class="wrap">
    <header>
      <p class="kicker">Southern OC + LA / OC / San Diego • events for teens & adults</p>
      <h1>Next 4 Weeks: Event Picks</h1>
      <div class="meta">
        <span class="pill"><span class="dot"></span><span><strong>Window:</strong> {escape(ws_label)} → {escape(we_label)}</span></span>
        <span class="pill"><span><strong>Total:</strong> {total_events} events</span></span>
        <span class="pill"><span><strong>Major Artists:</strong> {major_events}</span></span>
        <span class="pill"><span><strong>Generated:</strong> {escape(generated)}</span></span>
      </div>
      <div class="controls">
        <div class="search">
          <span style="color:var(--muted); font-size:13px;">Search</span>
          <input id="q" type="text" placeholder="Artist, show, venue, city…" />
        </div>
        <button class="btn" id="expandAll" type="button">Expand all</button>
        <button class="btn" id="collapseAll" type="button">Collapse all</button>
      </div>
      <nav class="nav">
        {"".join(week_nav)}
      </nav>
    </header>

    {"".join(week_sections)}

    <footer>
      Tip: “Major Artist” is based on a Wikipedia pageview popularity heuristic and may miss edge cases.
    </footer>
  </div>

  <script>{js}</script>
</body>
</html>"""
    return html

# Always write an HTML report (even if no events are found)
site_dir = _repo_root() / "site"
site_dir.mkdir(parents=True, exist_ok=True)
(site_dir / ".nojekyll").write_text("", encoding="utf-8")

out_path = site_dir / "index.html"
df_out = df.copy() if isinstance(df, pd.DataFrame) else pd.DataFrame()
out_path.write_text(make_html_report(df_out, START_DT, END_DT), encoding="utf-8")
print(f"Wrote: {out_path.resolve()} (events={len(df_out) if isinstance(df_out, pd.DataFrame) else 0})")
out_path


## 5) Scheduling (Tuesday runs)

**Option A (easiest, if you have Colab Pro/Pro+ Scheduled Notebooks):**
- Schedule the notebook to run **daily**, and add a guard cell that only proceeds on **Tuesdays**.  
  (Pro/Pro+ daily scheduling is the common limitation; running daily but exiting early is the workaround.)

**Option B (Colab Enterprise / Google Cloud):**
- Use **Colab Enterprise scheduled notebook runs** in the Google Cloud console to run on a **weekly** schedule.

**Option C (outside Colab):**
- Run the notebook via **papermill** in GitHub Actions / Cloud Run / cron and store the HTML report.
